## Jones matrix evolution with discrete scramblers along the link
This notebook models the change of the accumulated fibre Jones matrix with state of polarisation scramblers and/or slow drift.

We start by defining the system parameters as usual.

In [ ]:
from configparser import ConfigParser

parameters = ConfigParser()

parameters['FIBRE'] = {
    'correlation_length': '0.1',
    'beat_length': '0.05',
    'section_length': '0.167',
    'path_coordinates': '[ \
        [-73.12487396171335,-35.95811819864919], \
        [-72.8998741217013,-35.47096027456558] \
    ]',                          # Estimate coordinates of the Prat cable, taken from https://www.submarinecablemap.com/api/v3/cable/cable-geo.json
    'PMD_parameter':     '0.0',  # Polarisation mode dispersion parameter in ps / (km ^ 0.5)
    'realisation_count': '1',    # Number of fibre realisations to simulate simultaneously
    'photoelasticity':   '0.78', # Photoelasticity, which relates material strain to optical strain
}

Let's enable logging messages.

In [ ]:
import logging
logging.basicConfig(level = logging.INFO)

Now, we create the fibre and some scramblers.

In [ ]:
from tremor_waveplate_toolbox import FibreMarcuse, Scramblers

fibre = FibreMarcuse(parameters)
scramblers = Scramblers()

Request a scrambling perturbation:

In [ ]:
scramblers_perturbation = scramblers(
    path = fibre.section_path,
    vertex_indices = [100, 200], # We perturb the signal after the 100th and 200th step (with step size 0.167 m: 16.7 and 33.4 m after the transmitter)
    time_indices = [20, 40], # We perturb the signal at minutes 20 and 40
    sample_rate = 1 / 60, # The scramblers work on a clock with a resolution of 1 minute (sample rate of 1 / 60 Hz)
    sample_count = 60 # We obtain a scrambling signal for 60 samples / (1 / 60 Hz) = 3600 seconds (1 hour)
)

Once per second, we calculate the scrambler-perturbed fibre Jones matrix at the carrier frequency.

In [ ]:
from tqdm import tqdm
import numpy as np
try:
    import cupy as cp
except:
    logger.WARNING("No cupy installation found. For hardware acceleration, please follow the corresponding installation instructions")
    cp = np

transmission_start_times = np.arange(-60, 3661)
Jones_matrices = fibre.Jones(
    frequency_angular = cp.array([0]),
    carrier_wavelength = 1550.,
    transmission_start_times = transmission_start_times,
    perturbations = scramblers_perturbation,
    verbose = True
)

if cp != np: Jones_matrices = Jones_matrices.get() # Make sure Jones_matrices resides in CPU memory


Now, let's plot the results.
As a reminder: `Jones_matrices` has shape `[R, T, F, 2, 2]`, where `R` indexes the fibre realisation, `T` indexes the transmission start time in minutes, `F = 1` indexes the frequency (but we simulated only the carrier frequency, so `F = 1`), and the last two dimensions index the Jones matrix entries.

In [ ]:
import matplotlib.pyplot as plt

# Plot three realisations
fig, ax = plt.subplots(2, 2, figsize = (10, 5))
def plot_Jones_entry(row, column):
    ax[row, column].plot(transmission_start_times / 60, Jones_matrices[0, :, 0, row, column].real, color = 'tab:blue')
    ax[row, column].plot(transmission_start_times / 60, Jones_matrices[0, :, 0, row, column].imag, color = 'tab:orange')
    ax[row, column].set_xlabel("t [min]")
    ax[row, column].set_ylabel("$m_{" + str(row + 1) + str(column + 1) + "}$")
    ax[row, column].grid()
    ax[row, column].set_ylim([-1, 1])

for row in range(2):
    for column in range(2):
        plot_Jones_entry(row, column)

fig.tight_layout()

Let's repeat the whole thing, but with slow drift instead of discrete scramblers!

In [ ]:
from tremor_waveplate_toolbox import Drift

drift = Drift()
drift_perturbation = drift(
    path = fibre.section_path,
    drift_scalar = 5e-3,
    sample_rate = 1,
    sample_count = 3600
)

Jones_matrices = fibre.Jones(
    frequency_angular = cp.array([0]),
    carrier_wavelength = 1550.,
    transmission_start_times = transmission_start_times,
    perturbations = drift_perturbation,
    verbose = True
)

if cp != np: Jones_matrices = Jones_matrices.get()

fig, ax = plt.subplots(2, 2, figsize = (10, 5))

for row in range(2):
    for column in range(2):
        plot_Jones_entry(row, column)

fig.tight_layout()

Why choose between continuous drift and discrete scramblers? Let's model both at the same time!

In [ ]:
Jones_matrices = fibre.Jones(
    frequency_angular = cp.array([0]),
    carrier_wavelength = 1550.,
    transmission_start_times = transmission_start_times,
    perturbations = [drift_perturbation, scramblers_perturbation],
    verbose = True
)

if cp != np: Jones_matrices = Jones_matrices.get()

fig, ax = plt.subplots(2, 2, figsize = (10, 5))

for row in range(2):
    for column in range(2):
        plot_Jones_entry(row, column)

fig.tight_layout()

Note that the current version of the toolbox has simplified models for scramblers and drift, which are not physically accurate.
Also, notice that the Jones matrix is constant outside the range [0, 60], as the applied perturbations were limited to that range.
In other words, outside of the perturbations' defined scope, the model assumes the channel transfer matrix to return to its original, constant value.

So far, we've calculated only Jones transfer matrices of the channel.
Of course, we can also propagate signals through the same channel at different moments in time.

In [ ]:
parameters['TRANSCEIVER'] = {
    'constellation':    'QPSK',   # The symbol constellation to use
    'power':            '2',      # Transmission power in dBm
    'baud_rate':        '0.25e9', # Baud rate in symbols / s
    'pulse':            'RRCOS',  # Pulseshape, can be SINC or RRCOS, or define your own using the Pulse class
    'pulse_parameter':  '0.5',    # Parameter to pass to the pulse constructor. For a RRCOS pulse, this is the rolloff factor
    'upsample_factor':  '4',      # Samples per symbol
    'filter':           'RRCOS',  # Antialiasing (matched) filter
    'filter_parameter': '0.5'     # Same
}

parameters['SIGNAL'] = {
    'batch_size':   '1',   # The number of signals to transmit simultaneously
    'symbol_count': '20' # The number of symbols to transmit per signal
}

from tremor_waveplate_toolbox import Transmitter, Receiver, Device

transmitter = Transmitter(parameters)
symbols, signal = transmitter.transmit_random(
    parameters.getint('SIGNAL', 'batch_size'),
    int(parameters.getfloat('SIGNAL', 'symbol_count'))
)

if cp != np: signal.to_device(Device.CUDA)

transmission_start_times = np.array([10, 11, 30, 31, 50, 51]) * 60 # Transmit the same signal after 10, 30, and 50 minutes
signals_received = fibre(
    signal = signal,
    transmission_start_times = transmission_start_times,
    perturbations = [drift_perturbation, scramblers_perturbation],
    verbose = True
)

receiver = Receiver(parameters)
symbols_received = receiver(signals_received)

symbols_received.to_device(Device.CPU)

fig, ax = plt.subplots(2, 2, figsize = (10, 7))
def plot_symbols(time_index, row, column):
    symbols_to_plot = symbols_received.samples_time[0, time_index, :, column]
    ax[row, column].plot(symbols_received.time / 60, symbols_to_plot.real if row == 0 else symbols_to_plot.imag)
    if row == 0: ax[row, column].set_title(f"Polarisation {column + 1}")
    if row == 1: ax[row, column].set_xlabel("t - start time [min]")
    ax[row, column].grid(True)
    ax[row, column].set_xlim([0, symbols_received.duration / 60])
    ax[row, column].set_ylim([-np.sqrt(2), np.sqrt(2)])

legend_labels = []
for time_index, start_time in enumerate(transmission_start_times):
    for row in range(2):
        for column in range(2):
            plot_symbols(time_index, row, column)
    legend_labels.append(f"Transmitted @ {int(start_time / 60)} min")

ax[0, 0].set_ylabel("In-phase")
ax[1, 0].set_ylabel("Quadrature")

fig.legend(legend_labels)
fig.tight_layout()
